Question 1

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

# Define checkerboard size (number of inner corners per row and column)
CHECKERBOARD = (13, 22)  # Example: 14x23 inner corners

objp = np.zeros((13*22,3), np.float32)
objp[:,:2] = np.mgrid[0:13,0:22].T.reshape(-1,2)
# Arrays to store object points and image points from all the images.
objpoints = [] # 3d point in real world space
imgpoints = [] # 2d points in image plane.

# Directory containing the checkerboard images
DATA_DIR = "SelfCapture/Calibration/NoPattern"
image_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.jpeg")))  # Adjust extension if needed

# Function to detect and plot corners
def detect_and_plot_corners(image_path, plot_title, save_path=None):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load image: {image_path}")
        return False, None, img
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Detect checkerboard corners
    ret, corners=cv2.findChessboardCornersSB(gray, CHECKERBOARD, cv2.CALIB_CB_NORMALIZE_IMAGE,cv2.CALIB_CB_EXHAUSTIVE)

    if ret:
        print("ret found")
        objpoints.append(objp)
        # Refine corner positions
        corners = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), 
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30000, 0.0001))
        print(corners.shape)
        imgpoints.append(corners)
        # Draw corners on image for visualization
        img_with_corners = img.copy()
        cv2.drawChessboardCorners(img_with_corners, CHECKERBOARD, corners, ret)

        # Plot corners in pixel coordinates
        plt.figure(figsize=(8, 6))
        plt.imshow(img, cmap='gray')
        corners = corners.squeeze()  # Shape: (N, 2)
        plt.scatter(corners[:, 0], corners[:, 1], c='red', s=50, label='Detected Corners')
        plt.title(plot_title)
        plt.xlabel('Pixel Column (x)')
        plt.ylabel('Pixel Row (y)')
        plt.legend()
        if save_path:
            plt.savefig(save_path)
            plt.close()
        else:
            plt.show()

        return True, corners, img_with_corners
    else:
        # Plot the image even if detection failed
        plt.figure(figsize=(8, 6))
        plt.imshow(img, cmap='gray')
        plt.title(f"{plot_title} (No Corners Detected)")
        plt.xlabel('Pixel Column (x)')
        plt.ylabel('Pixel Row (y)')
        if save_path:
            plt.savefig(save_path)
            plt.close()
        else:
            plt.show()
        return False, None, img

# Part a: Detect and plot corners for the first image
if image_files:
    print("Processing first image...")
    success, corners, _ = detect_and_plot_corners(
        image_files[0], 
        "Detected Corners in First Checkerboard Image"
    )
    if success:
        print("Corners successfully detected in the first image.")
    else:
        print("Failed to detect corners in the first image.")

# Part b: Detect corners in all 30 images and report failures
print("\nProcessing all images...")
failed_images = []
for i, img_path in enumerate(image_files[:30]):  # Limit to 30 images
    print(f"\nProcessing image {i+1}: {os.path.basename(img_path)}")
    save_path = f"SelfCapture/output/output_image_{i+1}.jpeg"
    success, corners, _ = detect_and_plot_corners(
        img_path, 
        f"Checkerboard Image {i+1}", 
        save_path=save_path
    )
    if not success:
        print(f"Failed to detect corners in image {i+1}: {os.path.basename(img_path)}")
        failed_images.append((i+1, img_path, save_path))
    else:
        print(f"Successfully detected {len(corners)} corners in image {i+1}.")

# Report summary
print("\nSummary:")
print(f"Processed {min(len(image_files), 30)} images.")
print(f"Number of images with failed corner detection: {len(failed_images)}")
if failed_images:
    print("Images with failed corner detection:")
    for idx, img_path, save_path in failed_images:
        print(f"Image {idx}: {os.path.basename(img_path)} (see {save_path})")
else:
    print("All corners successfully detected in all images.")

img = cv2.imread("SelfCapture/Calibration/NoPattern/1.jpeg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
camera_width=img.shape[0]
camera_height=img.shape[1]
criteriaEnd = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30000, 0.0001)  # Termination criteria for estimateCameraParams
reprojectionError1, camera_matrix1, dist_coeffs1, rotation_vectors, translation_vectors = cv2.calibrateCamera(
    objpoints, imgpoints, (camera_width, camera_height), None, None,
    flags=cv2.CALIB_FIX_K4 + cv2.CALIB_FIX_K5 + cv2.CALIB_FIX_K6,criteria=criteriaEnd
)
print(reprojectionError1)

In [ ]:
# undistord 
newcameramtx, roi = cv2.getOptimalNewCameraMatrix(camera_matrix1, 
                                                  dist_coeffs1, 
                                                  (camera_width,camera_height), 
                                                  1, 
                                                  (camera_width,camera_height)
                                                  )


Question 2

In [ ]:
# load two undistorded images with custom feature points and calculate estrinsic 
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.0001)

# Load the image 1 
img = cv2.imread("SelfCapture/Cup/1.jpeg")
# Undistort the image
un_dst_img = cv2.undistort(img, camera_matrix1, dist_coeffs1, None, newcameramtx)
# Convert undistorted image to grayscale for corner detection
un_dst_gray = cv2.cvtColor(un_dst_img, cv2.COLOR_BGR2GRAY)
# Detect corners in the undistorted image
ret, corners_undist = cv2.findChessboardCornersSB(
    un_dst_gray, CHECKERBOARD, 
    flags=cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_EXHAUSTIVE
)
if ret:
    # Refine corners
    corners_undist = cv2.cornerSubPix(un_dst_gray, corners_undist, (11, 11), (-1, -1), criteria)
else:
    raise ValueError("Checkerboard corners not detected in undistorted image")

# Load the image 2
img_2 = cv2.imread("SelfCapture/Cup/2.jpeg")
# Undistort the image
un_dst_img_2 = cv2.undistort(img_2, camera_matrix1, dist_coeffs1, None, newcameramtx)
# Convert undistorted image to grayscale for corner detection
un_dst_gray_2 = cv2.cvtColor(un_dst_img_2, cv2.COLOR_BGR2GRAY)
# Detect corners in the undistorted image
ret, corners_undist = cv2.findChessboardCornersSB(
    un_dst_gray_2, CHECKERBOARD, 
    flags=cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_EXHAUSTIVE
)
if ret:
    # Refine corners
    corners_undist_2 = cv2.cornerSubPix(un_dst_gray_2, corners_undist, (11, 11), (-1, -1), criteria)
else:
    raise ValueError("Checkerboard corners not detected in undistorted image")



show markers in both images

In [ ]:
# we use thresholding for labeeling instead of label it manually
import cv2
import numpy as np

def detect_red_dots(image_filename):
    """
    Detect red dots in an image, label them from top-right to bottom-left, and return their center points.
    
    Args:
        image_filename (str): Path to the input image (e.g., 'image1.jpg').
    
    Returns:
        list: List of tuples (x, y) representing the center coordinates of detected dots.
    """
    # Load the image
    image = cv2.imread(image_filename)
    if image is None:
        raise ValueError(f"Could not load image: {image_filename}")

    # Convert to HSV color space
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Define range for red color, including darker reds
    lower_red1 = np.array([0, 70, 50])    # Lower bound for red (near 0 hue)
    upper_red1 = np.array([10, 255, 255]) # Upper bound for red
    lower_red2 = np.array([160, 70, 50])  # Lower bound for red (near 180 hue)
    upper_red2 = np.array([180, 255, 255]) # Upper bound for red

    # Create masks for red color
    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = mask1 + mask2  # Combine both red ranges

    # Apply morphological operations to clean up the mask
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)  # Remove small noise
    mask = cv2.dilate(mask, kernel, iterations=1) # Close small gaps

    # Find contours of the red dots
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Get centers of contours and sort by y then x (top-left to bottom-right)
    min_area = 16  # Minimum area for a dot
    max_area = 400
    y_min = 750
    dot_centers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area > min_area and area<max_area:
            (x, y), radius = cv2.minEnclosingCircle(contour)
            if y>y_min:
                dot_centers.append((int(x), int(y), radius))
    # Sort by y (ascending) then x (ascending)
    dot_centers.sort(key=lambda p: (p[1], -p[0]))

    # Draw contours and labels
    output_image = image.copy()
    for i, (x, y, radius) in enumerate(dot_centers):
        # Draw green circle around the dot
        cv2.circle(output_image, (x, y), int(radius), (0, 255, 0), 2)
        
        # Add larger label at top-right of the center
        label = f"Dot {i+1}"
        label_pos = (x + 5, y - 5)  # Top-right offset
        cv2.putText(output_image, label, label_pos, cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2, cv2.LINE_AA)

    # Save output image
    output_filename = image_filename.replace('.jpg', '_detected.jpg')
    cv2.imwrite(output_filename, output_image)

    # Return list of (x, y) coordinates
    return [(x, y) for x, y, _ in dot_centers]

# Example usage
if __name__ == "__main__":
    # Process multiple images
    image_files = ['SelfCapture/Cup/1.jpeg', 'SelfCapture/Cup/2.jpeg']
    points = detect_red_dots(image_files[0])
    print(f"Detected points in {points}:")
    points_2 = detect_red_dots(image_files[1])
    print(f"Detected points in {points_2}:")


In [ ]:
# calculate the extrinsic of both images now
# Intrinsic parameters
K = camera_matrix1

# Chessboard corners: 2D pixel coordinates (from both images)
img_points1 = corners_undist  # Shape: (N, 1, 2), e.g., from cv2.findChessboardCorners
img_points2 = corners_undist_2  # Shape: (N, 1, 2)

# 3D world points (15 mm spacing, 6x8 chessboard, Z=0)
square_size = 15.0  # mm
obj_points = np.zeros((13*22, 3), np.float32)
obj_points[:, :2] = np.mgrid[0:13, 0:22].T.reshape(-1, 2) * square_size
obj_points_h = cv2.convertPointsToHomogeneous(obj_points)[:, 0, :]  # Homogeneous: (N, 4)

# DLT function to estimate projection matrix
def dlt_projection_matrix(obj_points_h, img_points, K):
    # Normalize 2D points using intrinsic matrix
    img_points_h = cv2.convertPointsToHomogeneous(img_points)[:, 0, :]  # Shape: (N, 3)
    norm_points = (np.linalg.inv(K) @ img_points_h.T).T  # Shape: (N, 3)

    # Build the DLT matrix A
    A = []
    for (X, Y, Z, W), (x, y, w) in zip(obj_points_h, norm_points):
        A.append([X, Y, Z, W, 0, 0, 0, 0, -x*X, -x*Y, -x*Z, -x*W])
        A.append([0, 0, 0, 0, X, Y, Z, W, -y*X, -y*Y, -y*Z, -y*W])
    A = np.array(A)  # Shape: (2N, 12)

    # Solve using SVD
    _, _, Vh = np.linalg.svd(A)
    P = Vh[-1].reshape(3, 4)  # Last row of Vh (smallest singular value)
    P /= np.linalg.norm(P)  # Normalize for stability

    # Extract R, t
    Rt = np.linalg.inv(K) @ P  # [R | t]
    R = Rt[:, :3]
    t = Rt[:, 3]
    # Orthogonalize R
    U, _, Vt = np.linalg.svd(R)
    R = U @ Vt
    # Ensure det(R) = 1
    if np.linalg.det(R) < 0:
        R = -R
        t = -t
    return R, t

# Step 1: Estimate extrinsics using DLT
R1, t1 = dlt_projection_matrix(obj_points_h, img_points1, K)
R2, t2 = dlt_projection_matrix(obj_points_h, img_points2, K)

# Step 2: Compute relative pose
R = R2 @ R1.T
t = t2 - R @ t1

# Step 3: Set up projection matrices
P1 = K @ np.hstack((np.eye(3), np.zeros((3, 1))))  # Camera 1: [K | 0]
P2 = K @ np.hstack((R, t.reshape(3, 1)))           # Camera 2: [K | R, t]


In [ ]:
#  plotCamera() to visualize the camera position and orientations

Question 3

In [ ]:
# use undistorded image to perfome triangulation using DLT method

# Additional marker pixel coordinates (M markers) from cups 
marker_points1 = np.array([[u1_1, v1_1], [u1_2, v1_2], ...], dtype=np.float32)  # Image 1
marker_points2 = np.array([[u2_1, v2_1], [u2_2, v2_2], ...], dtype=np.float32)  # Image 2

# Step 4: Triangulate marker points
points1_h = cv2.convertPointsToHomogeneous(marker_points1)[:, 0, :]  # Shape: (M, 3)
points2_h = cv2.convertPointsToHomogeneous(marker_points2)[:, 0, :]  # Shape: (M, 3)
points_4d = cv2.triangulatePoints(P1, P2, points1_h.T, points2_h.T)  # Shape: (4, M)
points_3d = points_4d[:3] / points_4d[3]  # Normalize to Cartesian coordinates
points_3d = points_3d.T  # Shape: (M, 3), in camera 1 coordinates (mm)

# Step 5: Transform to world coordinate system (optional)
R1_inv = R1.T
t1_inv = -R1_inv @ t1
points_3d_world = (R1_inv @ points_3d.T + t1_inv).T  # Shape: (M, 3)

print("Reconstructed 3D points (world coordinates, mm):\n", points_3d_world)

# Verify reprojection error
points_3d_h = cv2.convertPointsToHomogeneous(points_3d_world)[:, 0, :]
proj_points1, _ = cv2.projectPoints(points_3d_h, cv2.Rodrigues(R1)[0], t1, K, np.zeros((5, 1)))
proj_points2, _ = cv2.projectPoints(points_3d_h, cv2.Rodrigues(R2)[0], t2, K, np.zeros((5, 1)))
reproj_error1 = np.mean(np.linalg.norm(proj_points1[:, 0, :] - marker_points1, axis=1))
reproj_error2 = np.mean(np.linalg.norm(proj_points2[:, 0, :] - marker_points2, axis=1))
print("Reprojection error (pixels): Image 1 =", reproj_error1, "Image 2 =", reproj_error2)

In [ ]:
# use undistorded image to perfome triangulation using PnP method
import cv2
import numpy as np

dist_coeffs = dist_coeffs1
# Additional marker pixel coordinates (N markers)
marker_points1 = np.array([[u1_1, v1_1], [u1_2, v1_2], ...], dtype=np.float32)  # Image 1
marker_points2 = np.array([[u2_1, v2_1], [u2_2, v2_2], ...], dtype=np.float32)  # Image 2

# Step 1: Estimate camera poses using PnP
success1, rvec1, tvec1 = cv2.solvePnP(obj_points, img_points1, K, dist_coeffs)
success2, rvec2, tvec2 = cv2.solvePnP(obj_points, img_points2, K, dist_coeffs)

# Convert rotation vectors to matrices
R1, _ = cv2.Rodrigues(rvec1)
R2, _ = cv2.Rodrigues(rvec2)

# Step 2: Compute relative pose
R = R2 @ R1.T
t = tvec2 - R @ tvec1

# Step 3: Set up projection matrices
P1 = K @ np.hstack((np.eye(3), np.zeros((3, 1))))  # Camera 1: [K | 0]
P2 = K @ np.hstack((R, t.reshape(3, 1)))           # Camera 2: [K | R, t]

# Step 4: Triangulate marker points
points1_h = cv2.convertPointsToHomogeneous(marker_points1)[:, 0, :]  # Shape: (N, 3)
points2_h = cv2.convertPointsToHomogeneous(marker_points2)[:, 0, :]  # Shape: (N, 3)
points_4d = cv2.triangulatePoints(P1, P2, points1_h.T, points2_h.T)  # Shape: (4, N)
points_3d = points_4d[:3] / points_4d[3]  # Normalize to Cartesian coordinates
points_3d = points_3d.T  # Shape: (N, 3), in camera 1 coordinates (mm)

# Step 5: Transform to world coordinate system (optional)
R1_inv = R1.T
t1_inv = -R1_inv @ tvec1
points_3d_world = (R1_inv @ points_3d.T + t1_inv).T  # Shape: (N, 3)

print("Reconstructed 3D points (world coordinates, mm):\n", points_3d_world)

# Step 6: Verify reprojection error
points_3d_h = cv2.convertPointsToHomogeneous(points_3d_world)[:, 0, :]  # Homogeneous
proj_points1, _ = cv2.projectPoints(points_3d_h, rvec1, tvec1, K, dist_coeffs)
proj_points2, _ = cv2.projectPoints(points_3d_h, rvec2, tvec2, K, dist_coeffs)
reproj_error1 = np.mean(np.linalg.norm(proj_points1[:, 0, :] - marker_points1, axis=1))
reproj_error2 = np.mean(np.linalg.norm(proj_points2[:, 0, :] - marker_points2, axis=1))
print("Reprojection error (pixels): Image 1 =", reproj_error1, "Image 2 =", reproj_error2)

Export point cloud as glb for question 5